# Lesson 2 — Run a policy yourself

Write and test a control loop for a pretrained ACT policy. First predict one action chunk from the current observation and send its first action. Then execute a bounded chunk prefix. Finally, run a complete episode that predicts again when it needs a new chunk.

The CPU checks use a deterministic environment. The required LIBERO experiment uses the saved ACT bundle and local simulator assets. The notebook also saves a video for the full episode.

LIBERO returns an observation and info on reset. Each step returns a new observation, reward, `terminated`, `truncated`, and info. `terminated` means the task ended. `truncated` means the time limit ended.


# Before you code

This course assumes strong Python skills and basic machine-learning knowledge. We explain PyTorch when it first appears. Read each shape, unit, mask, and parameter definition before you run the check. The reference copy gives a complete solution; the learner copy leaves the implementation for you.

The policy predicts a chunk Â ∈ R^(B×T×D), where B is the number of observations in a batch, T is the predicted time length in steps, and D is the number of controller values. For one LIBERO environment, B=1 and D=7. A control loop sends actions to the simulator one step at a time.

The observation path is: raw LIBERO camera and robot state → the LIBERO field adapter → the saved ACT preprocessor → prediction. The image processor converts uint8 pixels from [0,255] to float values in [0,1] by division by 255. It does not use the state mean and standard deviation. The preprocessor standardizes state values with mean μ and standard deviation σ fitted from training episodes: x_norm=(x−μ)/σ. The action postprocessor converts normalized predictions back with a_norm×σ_action+μ_action. Controller clipping occurs after that conversion and limits scaled controller commands to [-1,1]. These values are not metres or radians; LIBERO converts relative commands to simulator motion. The saved preprocessor and postprocessor store the fitted statistics and image operations.

Worked state example: x=0.7, μ=0.2, and σ=0.1 give x_norm=(0.7−0.2)/0.1=5. A normalized action 0.5 with μ_action=0.1 and σ_action=0.2 becomes 0.5×0.2+0.1=0.2 controller units, so clipping leaves it at 0.2. A converted value 1.4 becomes 1.0 after clipping.

There are two different limits. `max_steps` is a non-negative integer total for environment steps in one episode. `execute_steps` is a positive integer limit on actions from one predicted chunk. The actual chunk length and the remaining episode budget can reduce this limit. If the model predicts T=20 actions, `execute_steps=5` means use at most the first five. The environment returns a fresh observation after every action. The policy uses the latest observation when it makes its next prediction after the prefix, or earlier if the task ends.

LIBERO runs at 20 Hz: one relative action advances one simulator step, or 0.05 s. The converted dataset declares 10 FPS. This label does not establish the physical spacing of source control transitions. It does not set an action-repeat rule. We send each selected action once at the 20 Hz controller rate.

```text
observation → preprocess → predict (1,T,7) → choose a prefix
                                            ↓ one action per 20 Hz step
                         fresh observation ← environment.step(action)
                         stop at success, termination, truncation, or max_steps
```


In [1]:
import os
import sys
from collections.abc import Callable
from functools import partial
from pathlib import Path
from typing import TypedDict, TypeVar

import numpy as np
import torch
from numpy.typing import NDArray

from vla_lab.teaching.interfaces import (
    ActionChunkPolicy,
    ActionProcessor,
    EpisodeEnvironment,
    LiberoObservation,
    ObservationProcessor,
    TensorBatch,
)


def find_workspace(start: str | Path | None = None) -> Path:
    current = Path(start or Path.cwd()).resolve()
    if current.is_file():
        current = current.parent
    for folder in (current, *current.parents):
        if (folder / "manifests" / "act_references.yaml").is_file():
            return folder
    raise FileNotFoundError("Open this notebook inside the VLA course workspace.")


WORKSPACE = find_workspace()


def prepare_real_observation(
    observation: LiberoObservation, preprocessor: ObservationProcessor
) -> TensorBatch:
    """Apply the saved observation processor to one LIBERO observation."""
    return prepare_libero_observation(observation, preprocessor)


def process_controller_action(action: torch.Tensor, postprocessor: ActionProcessor) -> torch.Tensor:
    """Convert one normalized action to the controller action scale."""
    return postprocessor(action.unsqueeze(0))[0]


def predict_real_chunk(policy: ActionChunkPolicy, batch: TensorBatch) -> torch.Tensor:
    """Predict one action chunk from one prepared observation."""
    return policy.predict_action_chunk(batch)


if str(WORKSPACE / "src") not in sys.path:
    sys.path.insert(0, str(WORKSPACE / "src"))
HF_CACHE = WORKSPACE / ".local-data" / "hf"
(HF_CACHE / "hub").mkdir(parents=True, exist_ok=True)
(HF_CACHE / "datasets").mkdir(parents=True, exist_ok=True)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE / "hub")
os.environ["HF_DATASETS_CACHE"] = str(HF_CACHE / "datasets")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"


ObservationT = TypeVar("ObservationT")


class StepResult(TypedDict):
    observation: object
    reward: float
    success: bool
    terminated: bool
    truncated: bool
    action: NDArray[np.float32]


class PredictionStepResult(StepResult):
    chunk: torch.Tensor


class PrefixResult(TypedDict):
    observation: object
    steps: int
    done: bool
    success: bool
    terminated: bool
    truncated: bool
    stop_reason: str
    actions: list[NDArray[np.float32]]
    clipped_values: int


class EpisodeResult(TypedDict):
    success: bool
    steps: int
    policy_calls: int
    clipped_values: int
    stop_reason: str
    actions: list[NDArray[np.float32]]


def step_from_chunk(
    environment: EpisodeEnvironment[ObservationT],
    observation: ObservationT,
    chunk: torch.Tensor,
    action_to_controller: Callable[[torch.Tensor], torch.Tensor | NDArray[np.float32]],
) -> StepResult:
    """Apply the first predicted action and preserve the environment outcome."""
    if chunk.ndim != 3 or chunk.shape[0] != 1 or chunk.shape[2] != 7 or chunk.shape[1] < 1:
        raise ValueError("chunk must have a non-empty shape (1, T, 7)")
    if not torch.isfinite(chunk).all():
        raise ValueError("chunk actions must be finite")
    proposed = np.asarray(action_to_controller(chunk[0, 0]), dtype=np.float32).reshape(-1)
    if proposed.shape != (7,) or not np.isfinite(proposed).all():
        raise ValueError("controller action must contain seven finite values")
    action = np.clip(proposed, -1.0, 1.0).astype(np.float32, copy=False)
    next_observation, reward, terminated, truncated, info = environment.step(action)
    return {
        "observation": next_observation,
        "reward": reward,
        "success": bool(info.get("is_success", False)),
        "terminated": bool(terminated),
        "truncated": bool(truncated),
        "action": action.copy(),
    }

## Exercise 1 — Predict from an observation and step once

**Goal.** Prepare the current LIBERO observation, request one ACT chunk, convert its first action to controller units, and call `environment.step` once. The model input contains two camera tensors and robot state after preprocessing. The prediction has shape `(1,T,7)`; batch size is one, T is predicted action steps, and seven is the controller width.

Implement `predict_and_step`. Run the policy call without building a gradient graph. Preserve the environment reward, success flag, termination flag, truncation flag, and next observation. The supplied `step_from_chunk` helper performs one validated conversion and environment step. The next exercise reuses it for a chunk prefix.


In [2]:
class OneStepEnvironment:
    def __init__(self) -> None:
        self.action: NDArray[np.float32] | None = None

    def reset(self, seed: int | None = None) -> tuple[object, dict[str, int | None]]:
        return {"step": 0}, {"seed": seed}

    def step(self, action: np.ndarray) -> tuple[dict[str, int], float, bool, bool, dict[str, bool]]:
        self.action = np.asarray(action, dtype=np.float32)
        return {"step": 1}, 0.25, False, False, {"is_success": True}


def predict_and_step(
    environment: EpisodeEnvironment[ObservationT],
    policy: ActionChunkPolicy,
    observation: ObservationT,
    prepare_observation: Callable[[ObservationT], TensorBatch],
    predict_chunk: Callable[[ActionChunkPolicy, TensorBatch], torch.Tensor],
    action_to_controller: Callable[[torch.Tensor], torch.Tensor | NDArray[np.float32]],
) -> PredictionStepResult:
    """Predict from one observation, send the first action, and return both results."""
    obs_norm = prepare_observation(observation)
    with torch.no_grad():
        chunk = predict_chunk(policy, obs_norm).squeeze()
        chunk = chunk.clip(-1, 1)
    actions = action_to_controller(chunk)
    obs, reward, terminated, truncated, info = environment.step(actions)

    return PredictionStepResult(
        observation=obs,
        reward=reward,
        success=info["is_success"],
        terminated=terminated,
        truncated=truncated,
        action=actions,
    )


class OneChunkPolicy:
    def __init__(self) -> None:
        self.calls = 0

    def reset(self) -> None:
        return None

    def predict_action_chunk(self, batch: TensorBatch) -> torch.Tensor:
        self.calls += 1
        return torch.tensor([[[1.4, -1.2, 0, 0, 0, 0, 0]]])


environment = OneStepEnvironment()
policy = OneChunkPolicy()
step = predict_and_step(
    environment,
    policy,
    {"step": 0},
    lambda observation: {},
    predict_real_chunk,
    lambda action: action,
)
assert policy.calls == 1 and step["observation"] == {"step": 1}
assert step["success"] is True and step["terminated"] is False and step["truncated"] is False
assert np.array_equal(step["action"], np.array([1, -1, 0, 0, 0, 0, 0], dtype=np.float32)), step[
    "action"
]
print("One observation produced one chunk and one environment step.")

One observation produced one chunk and one environment step.


## Exercise 2 — Execute a chunk prefix and stop early

**Goal.** Execute up to a requested prefix of one predicted chunk. Three limits apply: the chunk length T, `execute_steps`, and the remaining episode budget `max_steps`. The function must use the smallest limit. After each environment step, it must stop at success, termination, or truncation.

For example, if T=20, `execute_steps=6`, and `max_steps=4`, execute four actions. If the environment reports success after action two, execute only two. This is the difference between a chunk prefix limit and a total episode step limit.

Implement `execute_chunk_prefix`. Return the latest observation, number of environment steps, whether the episode ended, success, stop reason, and the controller actions that were sent. The next exercise will request a new chunk only after this prefix ends without ending the episode.


In [35]:
def execute_chunk_prefix(
    environment: EpisodeEnvironment[ObservationT],
    observation: ObservationT,
    chunk: torch.Tensor,
    action_to_controller: Callable[[torch.Tensor], torch.Tensor | NDArray[np.float32]],
    *,
    execute_steps: int,
    max_steps: int,
) -> PrefixResult:
    """Execute a bounded prefix, stopping when the environment or step budget ends."""
    steps_todo = min(chunk.shape[1], execute_steps, max_steps)
    actions_sent = []
    clipped_values = 0

    done = False
    step = 0
    while step < steps_todo and not done:
        action_raw = chunk[0, step, :]
        action_raw = action_to_controller(action_raw)
        action = action_raw.clip(-1, 1)
        clipped_values += int(np.count_nonzero(action_raw != action))
        obs, reward, terminated, truncated, info = environment.step(action.numpy())
        actions_sent.append(action_raw.clone())

        step += 1
        done = terminated or truncated

    stop_reason = ""
    if terminated:
        if info["is_success"]:
            stop_reason = "success"
        else:
            stop_reason = "failure"
    elif truncated:
        stop_reason = "truncated"
    else:
        stop_reason = "step_limit"

    return PrefixResult(
        observation=obs,
        steps=step,
        done=done,
        success=info["is_success"],
        terminated=terminated,
        truncated=truncated,
        stop_reason=stop_reason,
        actions=actions_sent,
        clipped_values=clipped_values,
    )

In [37]:
class PrefixEnvironment:
    def __init__(self, end_after: int, end_kind: str = "success") -> None:
        self.end_after, self.end_kind = end_after, end_kind
        self.steps = 0
        self.actions: list[NDArray[np.float32]] = []
        self.seed: int | None = None

    def reset(self, seed: int | None = None) -> tuple[dict[str, int], dict[str, int | None]]:
        self.steps, self.actions, self.seed = 0, [], seed
        return {"step": 0}, {"seed": seed}

    def step(self, action: np.ndarray) -> tuple[dict[str, int], float, bool, bool, dict[str, bool]]:
        self.steps += 1
        action = np.asarray(action, dtype=np.float32)
        self.actions.append(action.copy())
        ended = self.steps >= self.end_after
        return (
            {"step": self.steps},
            0.0,
            ended and self.end_kind != "truncated",
            ended and self.end_kind == "truncated",
            {"is_success": ended and self.end_kind == "success"},
        )


class ChunkPolicyFixture:
    def __init__(self) -> None:
        self.calls = 0
        self.reset_count = 0

    def reset(self) -> None:
        self.reset_count += 1

    def predict_action_chunk(self, batch: TensorBatch) -> torch.Tensor:
        self.calls += 1
        return torch.full((1, 3, 7), 1.2)


fixture_env = PrefixEnvironment(end_after=2)
chunk = torch.full((1, 5, 7), 0.25)
prefix = execute_chunk_prefix(
    fixture_env,
    {"step": 0},
    chunk,
    lambda action: action,
    execute_steps=5,
    max_steps=5,
)
print(prefix)
assert prefix["steps"] == 2 and prefix["success"] is True and prefix["done"] is True
assert prefix["stop_reason"] == "success" and len(prefix["actions"]) == 2
budget_env = PrefixEnvironment(end_after=9)
budget = execute_chunk_prefix(
    budget_env,
    {"step": 0},
    chunk,
    lambda action: action,
    execute_steps=5,
    max_steps=3,
)
assert budget["steps"] == 3 and budget["done"] is False and budget["stop_reason"] == "step_limit"

{'observation': {'step': 2}, 'steps': 2, 'done': True, 'success': True, 'terminated': True, 'truncated': False, 'stop_reason': 'success', 'actions': [tensor([0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500]), tensor([0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500, 0.2500])], 'clipped_values': 0}


## Exercise 3 — Run a complete episode

**Goal.** Start from one reset observation. Predict a chunk, execute its allowed prefix, and keep the observation returned after every environment step. If the task continues, predict again from the latest observation. Stop at success, termination, truncation, or `max_steps`.

Here `max_steps=100` is the total episode budget. `execute_steps=10` limits each chunk prefix. If the model predicts 20 actions, one prediction can send at most 10 actions before the policy reads the latest observation and predicts again. The episode can use many chunks, but it cannot exceed 100 environment steps.

Implement `run_act_episode`. Return the success result, total environment steps, policy-call count, clipping count, stop reason, and sent actions. The required LIBERO run later uses this same function.

In [38]:
def run_act_episode(
    environment: EpisodeEnvironment[ObservationT],
    policy: ActionChunkPolicy,
    prepare_observation: Callable[[ObservationT], TensorBatch],
    predict_chunk: Callable[[ActionChunkPolicy, TensorBatch], torch.Tensor],
    action_to_controller: Callable[[torch.Tensor], torch.Tensor | NDArray[np.float32]],
    *,
    max_steps: int,
    execute_steps: int,
    seed: int | None = None,
) -> EpisodeResult:
    """Run one episode and report separate prediction and environment-step counts."""
    obs, info = environment.reset(seed)
    policy.reset()

    step = 0
    done = False
    policy_calls = 0
    clipped_values = 0
    all_actions: list[NDArray[np.float32]] = []

    while step < max_steps and not done:
        obs_norm = prepare_observation(obs)
        with torch.no_grad():
            chunk = predict_chunk(policy, obs_norm)
        prefix_result = execute_chunk_prefix(
            environment,
            obs_norm,
            chunk,
            action_to_controller,
            execute_steps=execute_steps,
            max_steps=max_steps - step,
        )

        obs = prefix_result["observation"]
        all_actions.extend(prefix_result["actions"])

        policy_calls += 1
        clipped_values += prefix_result["clipped_values"]
        step += prefix_result["steps"]
        done = prefix_result["done"]

    return EpisodeResult(
        success=prefix_result["success"],
        steps=step,
        policy_calls=policy_calls,
        clipped_values=clipped_values,
        stop_reason=prefix_result["stop_reason"],
        actions=all_actions,
    )


fixture_env = PrefixEnvironment(end_after=4)
fixture_policy = ChunkPolicyFixture()
fixture_result = run_act_episode(
    fixture_env,
    fixture_policy,
    lambda observation: {},
    lambda policy, batch: policy.predict_action_chunk(batch),
    lambda action: action,
    max_steps=4,
    execute_steps=2,
    seed=17,
)
assert fixture_result["steps"] == 4 and fixture_result["policy_calls"] == 2
assert fixture_result["success"] is True and fixture_result["stop_reason"] == "success"
assert fixture_policy.reset_count == 1 and fixture_env.seed == 17
assert fixture_result["clipped_values"] == 28
assert all(np.all(action == 1.0) for action in fixture_env.actions)
print("Episode check passed:", fixture_result)

Episode check passed: {'success': True, 'steps': 4, 'policy_calls': 2, 'clipped_values': 28, 'stop_reason': 'success', 'actions': [tensor([1.2000, 1.2000, 1.2000, 1.2000, 1.2000, 1.2000, 1.2000]), tensor([1.2000, 1.2000, 1.2000, 1.2000, 1.2000, 1.2000, 1.2000]), tensor([1.2000, 1.2000, 1.2000, 1.2000, 1.2000, 1.2000, 1.2000]), tensor([1.2000, 1.2000, 1.2000, 1.2000, 1.2000, 1.2000, 1.2000])]}


## Visual experiment — choose the chunk prefix

The slider changes only the number of actions allowed before ACT predicts again. The environment still returns an observation after every action; the policy does not use it until the next prediction. This drawing does not run a model or change a simulator.


In [30]:
import ipywidgets as widgets
from IPython.display import display

prefix_slider = widgets.IntSlider(value=5, min=1, max=20, step=1, description="execute_steps")
timeline = widgets.Output()


def draw_prefix(change: object | None = None) -> None:
    with timeline:
        timeline.clear_output(wait=True)
        used = prefix_slider.value
        print("Predicted actions:", " ".join("■" if i < used else "□" for i in range(20)))
        print(
            f"The environment returns an observation after every action. ACT requests a new prediction after at most {used} actions, unless the task ends first."
        )


prefix_slider.observe(draw_prefix, names="value")
display(prefix_slider, timeline)
draw_prefix()

IntSlider(value=5, description='execute_steps', max=20, min=1)

Output()

## Required LIBERO exercise — one prediction, one step

Use your episode loop and the prepared ACT bundle. This experiment requires the local checkpoint and LIBERO assets. It advances one action only. The group needs its declared local experiment dependencies.

In [39]:
from vla_lab.policy.bundle import load_act_bundle
from vla_lab.sim.episode import open_libero_task
from vla_lab.teaching.runtime import prepare_libero_observation

CHECKPOINT = WORKSPACE / ".local-data/runs/act-bowl-plate-probe-200/checkpoint-000200"

# Required LIBERO exercise: use your episode function to send one predicted action.
act_policy, act_preprocessor, act_postprocessor, manifest = load_act_bundle(CHECKPOINT)
act_policy.eval()
libero_environment = open_libero_task(
    "bowl_plate", 0, WORKSPACE / "configs", WORKSPACE / ".local-data"
)
try:
    result = run_act_episode(
        libero_environment,
        act_policy,
        partial(prepare_real_observation, preprocessor=act_preprocessor),
        predict_real_chunk,
        partial(process_controller_action, postprocessor=act_postprocessor),
        max_steps=1,
        execute_steps=1,
        seed=20260930,
    )
    print("One prediction and one LIBERO step:", result)
finally:
    libero_environment.close()

Loading weights from local directory
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
One prediction and one LIBERO step: {'success': np.False_, 'steps': 1, 'policy_calls': 1, 'clipped_values': 0, 'stop_reason': 'step_limit', 'actions': [tensor([ 0.9111,  0.1964, -0.1503, -0.0096,  0.0256, -0.0117, -0.8731])]}


## Run a complete LIBERO episode

The total episode limit is 100 environment steps. Each prediction can contribute at most 10 actions. The run saves a video under the simulator output folder. This experiment requires the one-step setup group.

In [40]:
# Complete LIBERO episode: same learner code, a total limit of 100 environment steps.
act_policy, act_preprocessor, act_postprocessor, manifest = load_act_bundle(CHECKPOINT)
act_policy.eval()
libero_environment = open_libero_task(
    "bowl_plate", 0, WORKSPACE / "configs", WORKSPACE / ".local-data"
)
try:
    result = run_act_episode(
        libero_environment,
        act_policy,
        partial(prepare_real_observation, preprocessor=act_preprocessor),
        predict_real_chunk,
        partial(process_controller_action, postprocessor=act_postprocessor),
        max_steps=100,
        execute_steps=10,
        seed=20260930,
    )
    print("Complete episode:", result)
finally:
    libero_environment.close()

Loading weights from local directory
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Complete episode: {'success': True, 'steps': 74, 'policy_calls': 8, 'clipped_values': 27, 'stop_reason': 'success', 'actions': [tensor([ 0.9111,  0.1964, -0.1503, -0.0096,  0.0256, -0.0117, -0.8731]), tensor([ 0.9225,  0.1986, -0.1442, -0.0097,  0.0256, -0.0117, -0.8830]), tensor([ 0.9379,  0.2025, -0.1334, -0.0097,  0.0263, -0.0118, -0.8922]), tensor([ 0.9392,  0.2034, -0.1309, -0.0097,  0.0264, -0.0117, -0.8955]), tensor([ 0.9437,  0.2044, -0.1292, -0.0099,  0.0265, -0.0118, -0.8972]), tensor([ 0.9193,  0.1972, -0.2005, -0.0091,  0.0241, -0.0123, -0.9100]), tensor([ 0.8962,  0.1911, -0.2638, -0.0087,  0.0218, -0.0128, -0.9255]), tensor([ 0.9184,  0.1971, -0.1925, -0.0092,  0.0243, -0.0123, -0.9074]), tensor([ 0.8981,  0.1915, -0.2637, -0.0087,  0.0218, -0.0128, -0.9274]), tensor([ 0.8099,  0.1706, -0.4467, -0.0075,  0.0147, -0.0143, -0.9610]), tensor([ 4.2566e-01,  7.7046e-02, -7.6312e-01, -5

## Experiment — compare chunk prefix limits

Compare `execute_steps` values 1, 5, and 10. Each trial uses the same development state and reset seed. Read success, environment steps, policy calls, and stop reason together. The storage rate of the dataset does not change the 20 Hz simulator action rate.

In [41]:
# Separate experiment: compare chunk prefix lengths on the same development state.
rows = []
for prefix_limit in (1, 5, 10):
    act_policy, act_preprocessor, act_postprocessor, manifest = load_act_bundle(CHECKPOINT)
    act_policy.eval()
    libero_environment = open_libero_task(
        "bowl_plate", 0, WORKSPACE / "configs", WORKSPACE / ".local-data"
    )
    try:
        result = run_act_episode(
            libero_environment,
            act_policy,
            partial(prepare_real_observation, preprocessor=act_preprocessor),
            predict_real_chunk,
            partial(process_controller_action, postprocessor=act_postprocessor),
            max_steps=100,
            execute_steps=prefix_limit,
            seed=20260930,
        )
        row = {
            "execute_steps": prefix_limit,
            **{k: result[k] for k in ("success", "steps", "policy_calls", "stop_reason")},
        }
        rows.append(row)
        print(row)
    finally:
        libero_environment.close()

Loading weights from local directory
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
{'execute_steps': 1, 'success': np.False_, 'steps': 100, 'policy_calls': 100, 'stop_reason': 'step_limit'}
Loading weights from local directory
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
{'execute_steps': 5, 'success': np.False_, 'steps': 100, 'policy_calls': 20, 'stop_reason': 'step_limit'}
Loading weights from local directory
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
{'execute_steps': 10, 'success': True, 'steps': 74, 'policy_calls': 8, 'stop_reason': 'success'}
